In [39]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime


month_mapping = {
    "1月": "Jan", "2月": "Feb", "3月": "Mar", "4月": "Apr",
    "5月": "May", "6月": "Jun", "7月": "Jul", "8月": "Aug",
    "9月": "Sep", "10月": "Oct", "11月": "Nov", "12月": "Dec"
}
time_pattern = r'(\d+月\s+\d+,\s+\d{4}\s+\d{1,2}:\d{2}:\d{2}\s+[上下]午)'

def parse_chinese_time(input_string):
    # 寻找匹配
    match = re.search(time_pattern, input_string)
    
    if not match:
        raise ValueError("无法在输入字符串中找到符合格式的时间")
    
    # 提取时间部分
    time_str = match.group(1)
    
    # 转换"上午"/"下午"为AM/PM
    if "上午" in time_str:
        time_str = time_str.replace("上午", "AM")
    elif "下午" in time_str:
        time_str = time_str.replace("下午", "PM")
    
    for cn_month, en_month in month_mapping.items():
        if cn_month in time_str:
            time_str = time_str.replace(cn_month, en_month)
            break
    
    # 解析时间字符串，格式如 "May 07, 2025 12:54:14 PM"
    dt = datetime.strptime(time_str, "%b %d, %Y %I:%M:%S %p")
    
    # 转换为时间戳（以秒为单位）
    timestamp = dt.timestamp()
    
    return timestamp


def read_log_file(file_path):
    lines = []
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            lines = file.readlines()
        lines = [line.rstrip('\n') for line in lines]
    except FileNotFoundError:
        print(f"错误: 文件 '{file_path}' 不存在")
    except Exception as e:
        print(f"读取文件时发生错误: {e}")
    return lines


# 定义四个模式
pattern1 = "信息: start dfs graph traversal"
pattern2 = "start LLM call on method judgment"
pattern3 = "信息: start LLM call on main question"
pattern4_regex = re.compile(r"信息: finish analyze： .*")

def find_pattern_sequences(lines):
    result_tuples = []
    i = 0
    
    while i < len(lines):
        # 寻找第一个模式
        if pattern1 in lines[i]:
            idx1 = i
            # 从当前位置继续寻找第二个模式
            j = i + 1
            while j < len(lines):
                if pattern2 in lines[j]:
                    idx2 = j
                    # 从当前位置继续寻找第三个模式
                    k = j + 1
                    while k < len(lines):
                        if pattern3 in lines[k]:
                            idx3 = k
                            # 从当前位置继续寻找第四个模式
                            m = k + 1
                            while m < len(lines):
                                if pattern4_regex.match(lines[m]):
                                    idx4 = m
                                    # 找到完整的模式序列，添加到结果中
                                    result_tuples.append((idx1 - 1, idx2 - 1, idx3 - 1, idx4 - 1))
                                    # 从第四个匹配后的位置继续查找新的序列
                                    i = idx4
                                    break
                                m += 1
                            break
                        k += 1
                    break
                j += 1
        i += 1
    
    return result_tuples


def get_times_from_log(lines, indexes):
    time0 = parse_chinese_time(lines[indexes[0]])
    time1 = parse_chinese_time(lines[indexes[1]])
    time2 = parse_chinese_time(lines[indexes[2]])
    time3 = parse_chinese_time(lines[indexes[3]])
    return [time1 - time0, time2 - time1, time3 - time2]


def analyze_with_trimming(data, trim_percent=2):
    """
    去除数据的最大和最小的指定百分比后，比较并打印截取前后的统计信息
    
    参数:
        data: 数字列表或数组
        trim_percent: 从两端去除的百分比数据，默认为1%
    
    返回:
        pandas.DataFrame: 包含截取前后统计数据比较的表格
    """
    # 转换为numpy数组
    arr = np.array(data)
    original_size = len(arr)
    
    # 计算截尾边界
    lower_bound = np.percentile(arr, trim_percent)
    upper_bound = np.percentile(arr, 100 - trim_percent)
    
    # 获取截尾后的数组
    trimmed_arr = arr[(arr >= lower_bound) & (arr <= upper_bound)]
    trimmed_size = len(trimmed_arr)
    
    # 计算基本统计量 - 原始数据
    original_stats = {
        "均值": np.mean(arr),
        "中位数": np.median(arr),
        "标准差": np.std(arr),
        "最小值": np.min(arr),
        "最大值": np.max(arr),
        "第一四分位数": np.percentile(arr, 25),
        "第三四分位数": np.percentile(arr, 75),
        "变异系数": np.std(arr) / np.mean(arr)
    }
    
    # 计算基本统计量 - 截尾后数据
    trimmed_stats = {
        "均值": np.mean(trimmed_arr),
        "中位数": np.median(trimmed_arr),
        "标准差": np.std(trimmed_arr),
        "最小值": np.min(trimmed_arr),
        "最大值": np.max(trimmed_arr),
        "第一四分位数": np.percentile(trimmed_arr, 25),
        "第三四分位数": np.percentile(trimmed_arr, 75),
        "变异系数": np.std(trimmed_arr) / np.mean(trimmed_arr)
    }
    
    # 打印截尾信息
    removed_count = original_size - trimmed_size
    print(f"数据截尾信息:")
    print(f"- 原始数据量: {original_size}")
    print(f"- 截尾后数据量: {trimmed_size}")
    print(f"- 去除了{removed_count}个极端值 (占总数据量的{removed_count/original_size*100:.2f}%)")
    print(f"- 去除的数值范围: <{lower_bound:.4f} 或 >{upper_bound:.4f}")
    
    # 将统计结果转换为DataFrame以便比较
    stats_df = pd.DataFrame({
        "原始数据": original_stats,
        f"截尾{trim_percent}%后": trimmed_stats,
        "变化百分比": {k: (trimmed_stats[k] - v) / v * 100 if v != 0 else float('inf') 
                      for k, v in original_stats.items()}
    })
    
    # 设置输出格式
    pd.set_option('display.float_format', '{:.4f}'.format)
    
    print("\n统计特征比较:")
    print(stats_df)
    
    return stats_df, (arr, trimmed_arr)


In [23]:


target_log_files = [
    "D:\\Workspace\\uncaught exception\\java-scanner\\ExperimentExecutor_Main_E0502.log",
    "D:\\Workspace\\uncaught exception\\java-scanner\\ExperimentExecutor_Main_E0401.log",
    "D:\\Workspace\\uncaught exception\\java-scanner\\ExperimentExecutor_Main_E0506-0.log"
    ]

process_times, analyse_times, first_call_times, main_call_times, total_times = [], [], [], [], []

def process_log_file_data(log_file_path):
    log_lines = read_log_file(log_file_path)
    target_tuples = find_pattern_sequences(log_lines)
    for t in target_tuples:
        process_time = get_times_from_log(log_lines, t)
        process_times.append(process_time)
        analyse_times.append(process_time[0])
        first_call_times.append(process_time[1])
        main_call_times.append(process_time[2])
        total_times.append(sum(process_time))

for file in target_log_files:
    process_log_file_data(file)


In [40]:
analyze_with_trimming(analyse_times)

数据截尾信息:
- 原始数据量: 1443
- 截尾后数据量: 1415
- 去除了28个极端值 (占总数据量的1.94%)
- 去除的数值范围: <0.0000 或 >69.0000

统计特征比较:
           原始数据   截尾2%后    变化百分比
均值       6.3860  3.2311 -49.4035
中位数      1.0000  1.0000   0.0000
标准差     26.3046  8.6034 -67.2933
最小值      0.0000  0.0000      inf
最大值    299.0000 69.0000 -76.9231
第一四分位数   0.0000  0.0000      inf
第三四分位数   2.0000  2.0000   0.0000
变异系数     4.1191  2.6627 -35.3578


(           原始数据   截尾2%后    变化百分比
 均值       6.3860  3.2311 -49.4035
 中位数      1.0000  1.0000   0.0000
 标准差     26.3046  8.6034 -67.2933
 最小值      0.0000  0.0000      inf
 最大值    299.0000 69.0000 -76.9231
 第一四分位数   0.0000  0.0000      inf
 第三四分位数   2.0000  2.0000   0.0000
 变异系数     4.1191  2.6627 -35.3578,
 (array([0., 0., 3., ..., 7., 0., 1.], shape=(1443,)),
  array([0., 0., 3., ..., 7., 0., 1.], shape=(1415,))))

In [41]:
analyze_with_trimming(first_call_times)

数据截尾信息:
- 原始数据量: 1443
- 截尾后数据量: 1414
- 去除了29个极端值 (占总数据量的2.01%)
- 去除的数值范围: <0.0000 或 >174.4000

统计特征比较:
           原始数据    截尾2%后    变化百分比
均值      19.0049  12.5000 -34.2273
中位数      0.0000   0.0000      inf
标准差     54.5922  20.9737 -61.5811
最小值      0.0000   0.0000      inf
最大值    864.0000 172.0000 -80.0926
第一四分位数   0.0000   0.0000      inf
第三四分位数  17.0000  16.0000  -5.8824
变异系数     2.8725   1.6779 -41.5884


(           原始数据    截尾2%后    变化百分比
 均值      19.0049  12.5000 -34.2273
 中位数      0.0000   0.0000      inf
 标准差     54.5922  20.9737 -61.5811
 最小值      0.0000   0.0000      inf
 最大值    864.0000 172.0000 -80.0926
 第一四分位数   0.0000   0.0000      inf
 第三四分位数  17.0000  16.0000  -5.8824
 变异系数     2.8725   1.6779 -41.5884,
 (array([ 0., 14., 29., ..., 37.,  0.,  0.], shape=(1443,)),
  array([ 0., 14., 29., ..., 37.,  0.,  0.], shape=(1414,))))

In [46]:
analyze_with_trimming(main_call_times)

数据截尾信息:
- 原始数据量: 1443
- 截尾后数据量: 1415
- 去除了28个极端值 (占总数据量的1.94%)
- 去除的数值范围: <0.0000 或 >44.0000

统计特征比较:
           原始数据   截尾2%后    变化百分比
均值      16.9328 16.1562  -4.5863
中位数     15.0000 15.0000   0.0000
标准差      9.5267  7.6274 -19.9363
最小值      0.0000  0.0000      inf
最大值    107.0000 44.0000 -58.8785
第一四分位数  11.0000 11.0000   0.0000
第三四分位数  20.0000 20.0000   0.0000
变异系数     0.5626  0.4721 -16.0878


(           原始数据   截尾2%后    变化百分比
 均值      16.9328 16.1562  -4.5863
 中位数     15.0000 15.0000   0.0000
 标准差      9.5267  7.6274 -19.9363
 最小值      0.0000  0.0000      inf
 最大值    107.0000 44.0000 -58.8785
 第一四分位数  11.0000 11.0000   0.0000
 第三四分位数  20.0000 20.0000   0.0000
 变异系数     0.5626  0.4721 -16.0878,
 (array([ 0., 20., 16., ..., 20., 31., 16.], shape=(1443,)),
  array([ 0., 20., 16., ..., 20., 31., 16.], shape=(1415,))))

In [55]:
analyze_with_trimming(total_times, 3)

数据截尾信息:
- 原始数据量: 1443
- 截尾后数据量: 1369
- 去除了74个极端值 (占总数据量的5.13%)
- 去除的数值范围: <6.0000 或 >192.9600

统计特征比较:
            原始数据    截尾3%后    变化百分比
均值       42.3236  32.6976 -22.7439
中位数      23.0000  23.0000   0.0000
标准差      73.8170  30.3539 -58.8796
最小值       0.0000   6.0000      inf
最大值    1040.0000 190.0000 -81.7308
第一四分位数   13.0000  13.0000   0.0000
第三四分位数   41.0000  39.0000  -4.8780
变异系数      1.7441   0.9283 -46.7739


(            原始数据    截尾3%后    变化百分比
 均值       42.3236  32.6976 -22.7439
 中位数      23.0000  23.0000   0.0000
 标准差      73.8170  30.3539 -58.8796
 最小值       0.0000   6.0000      inf
 最大值    1040.0000 190.0000 -81.7308
 第一四分位数   13.0000  13.0000   0.0000
 第三四分位数   41.0000  39.0000  -4.8780
 变异系数      1.7441   0.9283 -46.7739,
 (array([ 0., 34., 48., ..., 64., 31., 17.], shape=(1443,)),
  array([34., 48., 36., ..., 64., 31., 17.], shape=(1369,))))

qwen:
约2000* 6/5 = 2400 次实验
4017千tokens输出 单价0.0096元/千tokens 
8617千tokens输入 单价0.0024元/千tokens
共花费58.96元

2069次
3348千输出 32.14元
7181千输入 17.23元

deepseek：
约2000* 6/5 = 2400 次实验
共12736千tokens使用 
标准时间价格
百万tokens输入2元（缓存未命中）/0.5元（缓存命中）
百万tokens输出8元

ernie
1186次实验
4581次调用

